In [5]:
import pandas as pd

# 厳選10カラム
selected_features = [
    'Dst Port', 'Protocol', 'Flow Duration', 'Tot Fwd Pkts', 
    'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'Flow Byts/s', 
    'Flow Pkts/s', 'Init Fwd Win Byts', 'Idle Mean'
]

def serialize_selected_logs(row):
    """
    選定した10列を結合してテキスト化する
    """
    # 数値が科学的表記(1e+05等)にならないよう、floatを整形
    parts = []
    for col in selected_features:
        val = row[col]
        # float型で整数に近い場合は整数表示、そうでない場合は小数点以下2桁程度にする
        if isinstance(val, float):
            val = f"{val:.2f}" if not val.is_integer() else f"{int(val)}"
        parts.append(f"{col}: {val}")
    
    return " | ".join(parts)

# データの読み込み（前回のCSV）
k_df = pd.read_csv("../data/processed/cicids/knowledge_benign_100.csv")
t_df = pd.read_csv("../data/processed/cicids/test_imbalanced_100.csv")

# 10列絞り込みバージョンのテキスト作成
k_df['text'] = k_df.apply(serialize_selected_logs, axis=1)
t_df['text'] = t_df.apply(serialize_selected_logs, axis=1)

# 文字数を確認
print(f"絞り込み後の1行あたりの平均文字数: {k_df['text'].str.len().mean():.1f} 文字")
print("\n--- 生成されたテキストの例 ---")
print(k_df['text'].iloc[0])

絞り込み後の1行あたりの平均文字数: 193.3 文字

--- 生成されたテキストの例 ---
Dst Port: 16703 | Protocol: 6 | Flow Duration: 34 | Tot Fwd Pkts: 2 | Tot Bwd Pkts: 0 | TotLen Fwd Pkts: 0 | Flow Byts/s: 0 | Flow Pkts/s: 58823.53 | Init Fwd Win Byts: 64240 | Idle Mean: 0


In [6]:
# 'text' 列（10列絞り込み後のテキスト）の長さを計算
k_df['text_len'] = k_df['text'].str.len()

# 最大長のインデックスを取得
max_idx = k_df['text_len'].idxmax()
max_len = k_df['text_len'].max()

print(f"最大文字列長: {max_len} 文字")
print(f"インデックス: {max_idx}")
print("-" * 30)
print("--- 最大長のログ内容 ---")
print(k_df.loc[max_idx, 'text'])

最大文字列長: 211 文字
インデックス: 27
------------------------------
--- 最大長のログ内容 ---
Dst Port: 443 | Protocol: 6 | Flow Duration: 68834686 | Tot Fwd Pkts: 253 | Tot Bwd Pkts: 419 | TotLen Fwd Pkts: 3663 | Flow Byts/s: 8610.86 | Flow Pkts/s: 9.76 | Init Fwd Win Byts: 8192 | Idle Mean: 10033256.33


In [7]:
import os

# 保存先ディレクトリの作成
processed_dir = "../data/processed/cicids"
os.makedirs(processed_dir, exist_ok=True)

# 1. 知識用データ（DB登録用）: Label と text のみ抽出
knowledge_final_df = k_df[['Label', 'text']].copy()
knowledge_final_path = os.path.join(processed_dir, "knowledge_rag_100.csv")
knowledge_final_df.to_csv(knowledge_final_path, index=False)

# 2. テスト用データ（評価用）: Label と text のみ抽出
test_final_df = t_df[['Label', 'text']].copy()
test_final_path = os.path.join(processed_dir, "test_rag_100.csv")
test_final_df.to_csv(test_final_path, index=False)

print(f"永続化完了:\n- {knowledge_final_path} ({len(knowledge_final_df)}件)\n- {test_final_path} ({len(test_final_df)}件)")

永続化完了:
- ../data/processed/cicids/knowledge_rag_100.csv (100件)
- ../data/processed/cicids/test_rag_100.csv (100件)
